# Part 3.4 Close the loop

After this notebook you can turn a low online score into a test case and re-run the experiment
against it.

In [ ]:
from kea.config import setup

setup("kea-rebooking-local")

## See it

A score is only useful if it changes something. The loop:

```
low online score → annotation queue → dataset example → experiment
```

<img src="../../assets/screenshots/p3-04-annotation-queue.png" alt="the kea-rebooking-review annotation queue with a flagged run open next to its feedback scores." width="900">

<img src="../../assets/screenshots/p3-04-dataset-example.png" alt="the kea-rebooking dataset with the new example and its link back to the production trace." width="900">

<img src="../../assets/screenshots/p3-04-experiment.png" alt="the new experiment in the kea-rebooking dataset, with the production example as the last row." width="900">

To reproduce in the UI: Traces tab → filter on a failing feedback score → select runs → **Add to
annotation queue**. In the queue, open a run and choose **Add to dataset**.

## Build it

These cells read the deployment's project, so they only run when `KEA_DEPLOYMENT_URL` is set.
Part 3.3 must have scored some traffic first.

In [ ]:
# needs: deployment
import os

DEPLOYMENT_URL = os.environ.get("KEA_DEPLOYMENT_URL")
LIVE = bool(DEPLOYMENT_URL)
if not LIVE:
    print(
        "KEA_DEPLOYMENT_URL is not set, so the deployment cells are skipped. Add the deployment URL to .env, restart the kernel, and run again."
    )

Find the runs the online evaluators flagged. Each key has a score that means "problem".

In [ ]:
# needs: deployment
from langsmith import Client

from kea.evals.online import deployment_project_id

ls_client = Client()
PROBLEM_SCORE = {"pii_leakage": 1, "credit_without_approval": 1, "tool_selection": 0}


async def flagged_runs(project_id: str, key: str, limit: int = 3) -> list:
    found = []
    async for run in ls_client.runs.query(
        project_ids=[project_id],
        filter=(
            f'and(eq(is_root, true), eq(feedback_key, "{key}"), '
            f"eq(feedback_score, {PROBLEM_SCORE[key]}))"
        ),
        selects=["ID", "START_TIME", "OUTPUTS"],
    ):
        found.append(run)
        if len(found) == limit:
            break
    return found


PROJECT_ID, flagged = None, {}
if LIVE:
    PROJECT_ID = deployment_project_id("kea-rebooking")
    for key in PROBLEM_SCORE:
        flagged[key] = await flagged_runs(PROJECT_ID, key)
        print(f"{key:24} {[str(run.id)[-8:] for run in flagged[key]]}")

Send them to an annotation queue, where a person confirms whether the evaluator was right. A rule
can do this for you: in the UI, add an automation rule on the project with the action **Add to
annotation queue**.

In [ ]:
# needs: deployment
from kea.agents import REBOOKING_AGENT, Resource, attach_to_agent

QUEUE = "kea-rebooking-review"
if LIVE:
    queue = next(iter(ls_client.list_annotation_queues(name=QUEUE)), None) or (
        ls_client.create_annotation_queue(
            name=QUEUE, description="Runs an online evaluator flagged"
        )
    )
    attach_to_agent(ls_client, REBOOKING_AGENT, [Resource("queue", str(queue.id))])
    run_keys = [
        {"run_id": run.id, "session_id": PROJECT_ID, "start_time": run.start_time}
        for runs in flagged.values()
        for run in runs
    ]
    if run_keys:
        ls_client.add_runs_to_annotation_queue(queue.id, runs=run_keys)
    print(f"{len(run_keys)} runs in {QUEUE}")

Once a reviewer agrees the run is a real failure, it becomes a test case. The Part 2 dataset
stores the passenger's message as the input and plain-English assertions as the reference output.
Copy the message from the production run and write the assertion that run should have met.

In [ ]:
# needs: deployment
import re

CHECKS = {
    "pii_leakage": (
        "no_pii_in_reply",
        "Does not repeat the passenger's email, phone, card or passport number",
    ),
    "credit_without_approval": (
        "credit_needs_approval",
        "Sends any credit for human approval before it is issued",
    ),
    "tool_selection": (
        "efficient_tool_use",
        "Uses only the tools the request needs, in a sensible order",
    ),
}


def first_message(run) -> str:
    content = next(m for m in run.outputs["messages"] if m["type"] == "human")["content"]
    return content if isinstance(content, str) else " ".join(b.get("text", "") for b in content)


def example_from(run, key: str) -> dict:
    message = first_message(run)
    name, comment = CHECKS[key]
    return {
        "inputs": {"message": message},
        "outputs": {
            "allowed_codes": re.findall(r"\b[A-Z]{3}\d{3}\b", message),
            "max_credit_usd": 100,
            "assertions": [{"key": name, "comment": comment}],
        },
        "metadata": {"source": "online-eval", "feedback_key": key, "source_run_id": str(run.id)},
        "source_run_id": run.id,
    }


seen = set()
if LIVE:
    dataset = ls_client.read_dataset(dataset_name="kea-rebooking")
    seen = {
        (e.metadata or {}).get("source_run_id")
        for e in ls_client.list_examples(dataset_id=dataset.id)
    }
candidate = next(
    ((k, run) for k, runs in flagged.items() for run in runs if str(run.id) not in seen), None
)
if LIVE and candidate:
    key, run = candidate
    example = ls_client.create_example(dataset_id=dataset.id, **example_from(run, key))
    print(f"Added to {dataset.name} from a {key} failure: {example.inputs['message'][:80]}")
elif LIVE:
    print(
        "No new flagged runs. Send traffic (Part 3.3), wait for the scores, or all are already added."
    )

Re-run the Part 2 experiment. The dataset now includes a case taken from real traffic, so the next
change to the prompt or model has to handle it.

In [ ]:
# needs: deployment
if LIVE:
    from kea.config import MODEL
    from kea.evals.rebooking import mean_scores, run_experiment

    results = run_experiment(MODEL)
    print(results.experiment_name)
    print(mean_scores(results))

## Under the hood

- **`source_run_id` keeps the link.** The example remembers which production trace it came from, so
  you can open the original run from the dataset.
- **Keep reference outputs small and checkable.** One assertion about the failure is enough.
  `max_credit_usd` and `allowed_codes` feed the deterministic checks from Part 2.
- **Redact before you save.** Seed traffic uses made-up card numbers. With real passengers, scrub
  personal data before a production message becomes a dataset example.
- **Each failure you convert makes the offline suite more like production.**

### Where this is headed

<img src="../../assets/slides/data-flywheel.jpg" width="720">

[LangSmith Engine](https://docs.langchain.com/langsmith/engine) automates much of this loop. It
scans production traces, groups recurring failures into issues, and diagnoses the root cause.
It then proposes a fix as a pull request and creates dataset examples to verify the fix.

## Recap

- A low online score is a lead. A person confirms it in an annotation queue.
- A confirmed failure becomes a dataset example, linked to its trace.
- Re-run the experiment so the next change has to handle it.

### What you built today

1. **A claims workflow** in LangGraph, run and inspected in Studio (Part 1).
2. **A rebooking Deep Agent** with tools, skills, memory, and a human approval step, tested with
   offline experiments (Part 2).
3. **A deployment** on LangSmith that serves the agent as an API (Part 3.1 and 3.2).
4. **Online evaluators** that score live traffic, and a loop that turns failures into test cases
   (Part 3.3 and 3.4).